# Tratamiento de los datos para las pruebas

**Qué hace este notebook.** Toma `Datos/datos_procesados/base_limpia.csv` y convierte las variables categóricas (las que son texto, como "Hombre" o "Rural") en columnas de **0 y 1**. A esto se le llama *one-hot*. Los modelos de regresión no entienden texto, pero sí números, y con las columnas de 0 y 1 pueden usar cada categoría como una variable externa.

**Ejemplo.** Una fila con `SEXO = "Mujer"` y `TIPOSEC = "Rural"` queda con `SEXO_Mujer = 1`, `SEXO_Hombre = 0`, `SEC_Rural = 1`, `SEC_Urbano = 0` y `SEC_Mixto = 0`. En cada variable, exactamente una columna vale 1.

**Qué no hace.** Los registros se quedan **como están** (una fila por grupo de sexo, edad y tipo de sección, en cada municipio y elección, igual que en la base limpia): no se agrega nada y no se calculan porcentajes ni personas. Esos cálculos los hace cada prueba cuando los necesita.

**Resultado.** Se guarda `Datos/datos_procesados/base_tratada.csv`, el único archivo que cargan los notebooks de `notebooks/pruebas/`.

Variables a las que se aplica el one-hot:

| Columna original | Prefijo | Columnas nuevas |
|---|---|---|
| `TIPO_ELECCION` | `TIPO_` | Presidencial, Intermedia |
| `SEXO` | `SEXO_` | Hombre, Mujer |
| `RANGO_EDAD` | `EDAD_` | 8 rangos de edad (18-29, 30-39, … , 90 o más) |
| `TIPOSEC` | `SEC_` | Urbano, Rural, Mixto |

`ESTADO` y `MUNICIPIO` se dejan como texto: sirven para identificar a cada municipio (con 32 estados y 2,475 municipios, convertirlos en columnas de 0 y 1 agregaría casi 2,500 columnas, y dentro de un municipio serían constantes). `AELEC` (el año de la elección) tampoco se transforma.

## 1. Configuración y lectura
Se define la ruta de la base limpia (entrada) y la de la base tratada (salida), y se lee la base. Se guardan el número de filas (`FILAS`) y la lista de columnas originales (`COLUMNAS`) para poder comprobar al final que los registros no cambiaron.

In [9]:
from pathlib import Path
import pandas as pd

_cwd = Path.cwd().resolve()
RAIZ = next(p for p in [_cwd, *_cwd.parents] if (p / "Datos").is_dir())
RUTA_BASE = RAIZ / "Datos" / "datos_procesados" / "base_limpia.csv"
RUTA_SALIDA = RAIZ / "Datos" / "datos_procesados" / "base_tratada.csv"

base = pd.read_csv(RUTA_BASE, encoding="utf-8-sig")
FILAS = len(base)
COLUMNAS = list(base.columns)
print(f"{FILAS:,} filas | {len(COLUMNAS)} columnas")
base.head()

540,024 filas | 13 columnas


,AELEC,FELECCION,TIPO_ELECCION,ESTADO,MUNICIPIO,SEXO,RANGO_EDAD,TIPOSEC,SV,NV,NS,ABSTENCION,CALIDAD_NS
0,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Mixto,765,2544,3512,0.7688,0.5149
1,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Rural,1002,2982,0,0.7485,0.0000
2,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Urbano,17777,45137,9191,0.7174,0.1275
3,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Mixto,723,1580,2814,0.6861,0.5499
4,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Rural,1067,2074,0,0.6603,0.0000


## 2. One-hot
Para cada variable categórica de `VARIABLES` se crea una columna de 0 y 1 por categoría con `pd.get_dummies` y se agrega al final de la base. El prefijo indica de qué variable viene cada columna (por ejemplo `SEXO_Mujer`). Se usa el tipo `int8` (un entero muy pequeño) para que el archivo no ocupe más de lo necesario.

In [10]:
VARIABLES = {"TIPO_ELECCION": "TIPO", "SEXO": "SEXO", "RANGO_EDAD": "EDAD", "TIPOSEC": "SEC"}

for columna, prefijo in VARIABLES.items():
    dummies = pd.get_dummies(base[columna], prefix=prefijo, dtype="int8")
    base = pd.concat([base, dummies], axis=1)
    print(f"{columna}: {dummies.shape[1]} columnas nuevas")

print(f"\nAhora son {base.shape[1]} columnas")
base.head()

TIPO_ELECCION: 2 columnas nuevas
SEXO: 2 columnas nuevas
RANGO_EDAD: 8 columnas nuevas
TIPOSEC: 3 columnas nuevas

Ahora son 28 columnas


,AELEC,FELECCION,TIPO_ELECCION,ESTADO,MUNICIPIO,SEXO,RANGO_EDAD,TIPOSEC,SV,NV,...,EDAD_30-39,EDAD_40-49,EDAD_50-59,EDAD_60-69,EDAD_70-79,EDAD_80-89,EDAD_90 o más,SEC_Mixto,SEC_Rural,SEC_Urbano
0,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Mixto,765,2544,...,0,0,0,0,0,0,0,1,0,0
1,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Rural,1002,2982,...,0,0,0,0,0,0,0,0,1,0
2,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,18-29,Urbano,17777,45137,...,0,0,0,0,0,0,0,0,0,1
3,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Mixto,723,1580,...,1,0,0,0,0,0,0,1,0,0
4,2009,05/07/2009,Intermedia,AGUASCALIENTES,AGUASCALIENTES,Hombre,30-39,Rural,1067,2074,...,1,0,0,0,0,0,0,0,1,0


## 3. Eliminación de variables redundantes
Una vez creadas las columnas de 0 y 1, el texto original ya no aporta información nueva: `SEXO = "Mujer"` es lo mismo que `SEXO_Mujer = 1`. Por eso se eliminan las columnas originales de `VARIABLES`, y la base queda más ligera y solo con números en esas variables. `ESTADO`, `MUNICIPIO` y `AELEC` se conservan como identificadores.

In [15]:
base.drop(columns=list(VARIABLES.keys()), inplace=True)

## 4. Revisión
Se revisa la estructura de la base con `base.info()`: cuántas filas y columnas tiene, el tipo de cada columna y que no haya valores vacíos (`Non-Null Count` igual al número de filas en todas las columnas). Las filas deben ser las mismas que en la base limpia, las columnas de 0 y 1 deben ser de tipo `int8` y las variables eliminadas ya no deben aparecer.

In [16]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540024 entries, 0 to 540023
Data columns (total 24 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   AELEC              540024 non-null  int64  
 1   FELECCION          540024 non-null  object 
 2   ESTADO             540024 non-null  object 
 3   MUNICIPIO          540024 non-null  object 
 4   SV                 540024 non-null  int64  
 5   NV                 540024 non-null  int64  
 6   NS                 540024 non-null  int64  
 7   ABSTENCION         540024 non-null  float64
 8   CALIDAD_NS         540024 non-null  float64
 9   TIPO_Intermedia    540024 non-null  int8   
 10  TIPO_Presidencial  540024 non-null  int8   
 11  SEXO_Hombre        540024 non-null  int8   
 12  SEXO_Mujer         540024 non-null  int8   
 13  EDAD_18-29         540024 non-null  int8   
 14  EDAD_30-39         540024 non-null  int8   
 15  EDAD_40-49         540024 non-null  int8   
 16  ED

## 5. Guardar
Se guarda la base en `Datos/datos_procesados/base_tratada.csv` con la codificación `utf-8-sig` (la misma que usa la base limpia, para que los acentos se lean bien en Excel). Al final se imprime el tamaño del archivo y cuántos registros tiene cada columna nueva, como una última comprobación.

In [17]:
base.to_csv(RUTA_SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado: {RUTA_SALIDA} | {len(base):,} filas | {base.shape[1]} columnas | {RUTA_SALIDA.stat().st_size / 1e6:.0f} MB")
base.iloc[:, len(COLUMNAS):].sum().head(10)

Guardado: C:\Users\camil\Documents\Programming\INE\Datos\datos_procesados\base_tratada.csv | 540,024 filas | 24 columnas | 49 MB


EDAD_18-29        67801
EDAD_30-39        67804
EDAD_40-49        67804
EDAD_50-59        67804
EDAD_60-69        67798
EDAD_70-79        67793
EDAD_80-89        67691
EDAD_90 o más     65529
SEC_Mixto        194661
SEC_Rural        185410
dtype: int64